# 05 - Agente Completo

En este notebook final integramos todo lo aprendido:
- Modelo local con Ollama
- Herramientas personalizadas
- RAG con documentos locales
- Memoria conversacional

Construiremos un **agente investigador** que puede buscar en documentos, hacer cálculos y recordar la conversación.

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
import math
import datetime

# Modelos
llm = ChatOllama(model="gemma3:12b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

## 5.1 Preparar la base de conocimiento

In [ ]:
# Base de conocimiento de ejemplo
documentos = [
    Document(page_content="""El conflicto armado interno colombiano duró más de 50 años e involucró 
    a múltiples actores: guerrillas (FARC-EP, ELN, EPL), paramilitares (AUC), fuerzas del Estado, 
    y bandas criminales. Se estima que dejó más de 9 millones de víctimas registradas en el 
    Registro Único de Víctimas (RUV).""", metadata={"tema": "conflicto"}),
    Document(page_content="""La desaparición forzada es uno de los crímenes más devastadores del 
    conflicto colombiano. Según el Observatorio de Memoria y Conflicto, se registran más de 
    80.000 personas desaparecidas forzadamente. La UBPD trabaja en la búsqueda humanitaria 
    de estas personas.""", metadata={"tema": "desaparicion"}),
    Document(page_content="""Las versiones voluntarias son declaraciones que excombatientes 
    entregan de manera voluntaria ante la JEP o la UBPD, proporcionando información sobre 
    hechos del conflicto armado, incluyendo la ubicación de personas desaparecidas y las 
    circunstancias de los hechos.""", metadata={"tema": "versiones"}),
    Document(page_content="""La justicia transicional busca equilibrar las demandas de justicia 
    con la necesidad de paz. En Colombia, el SIVJRNR combina mecanismos judiciales (JEP) 
    con extrajudiciales (Comisión de la Verdad, UBPD) para garantizar verdad, justicia, 
    reparación y no repetición.""", metadata={"tema": "justicia"}),
    Document(page_content="""Las técnicas de procesamiento de lenguaje natural (NLP) se están 
    usando cada vez más en contextos humanitarios. La codificación cualitativa de entrevistas, 
    tradicionalmente hecha a mano con herramientas como NVivo, puede ser asistida por modelos 
    de lenguaje para acelerar el proceso manteniendo la calidad.""", metadata={"tema": "tecnologia"}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
fragmentos = splitter.split_documents(documentos)

vectorstore = FAISS.from_documents(
    documents=fragmentos,
    embedding=embeddings,
    
)

print(f"Base de conocimiento lista: {vectorstore.index.ntotal} fragmentos")

## 5.2 Definir las herramientas del agente

In [ ]:
@tool
def buscar_documentos(consulta: str) -> str:
    """Busca información relevante en la base de documentos. Úsala cuando 
    necesites datos específicos sobre el conflicto, desaparición forzada, 
    justicia transicional o la UBPD."""
    resultados = vectorstore.similarity_search(consulta, k=3)
    if not resultados:
        return "No encontré documentos relevantes."
    textos = []
    for i, doc in enumerate(resultados):
        tema = doc.metadata.get('tema', 'N/A')
        textos.append(f"[{i+1}] (tema: {tema}) {doc.page_content}")
    return "\n\n".join(textos)


@tool
def calculadora(expresion: str) -> str:
    """Evalúa expresiones matemáticas. Ejemplo: '1500 * 3.5 / 60'"""
    try:
        resultado = eval(expresion, {"math": math, "__builtins__": {}})
        return f"Resultado: {resultado}"
    except Exception as e:
        return f"Error: {e}"


@tool
def fecha_hora() -> str:
    """Devuelve la fecha y hora actual."""
    return datetime.datetime.now().strftime("%A %d de %B de %Y, %H:%M")


@tool
def generar_resumen(texto: str) -> str:
    """Genera un resumen estructurado de un texto largo. Úsala cuando necesites 
    condensar información de múltiples fuentes."""
    prompt = f"Resume el siguiente texto en 3 puntos clave, en español:\n\n{texto}"
    respuesta = llm.invoke(prompt)
    return respuesta.content


herramientas = [buscar_documentos, calculadora, fecha_hora, generar_resumen]
print("Herramientas listas:", [h.name for h in herramientas])

## 5.3 Crear el agente completo

In [ ]:
PROMPT_SISTEMA = """Eres un asistente investigador especializado en el conflicto armado 
colombiano y la búsqueda de personas desaparecidas.

Tu comportamiento:
1. Cuando te hagan preguntas sobre el conflicto, justicia transicional o desaparición 
   forzada, SIEMPRE busca primero en los documentos disponibles.
2. Para cálculos numéricos, usa la calculadora.
3. Si te piden resumir información, usa la herramienta de resumen.
4. Basa tus respuestas en la información encontrada. Si no hay información suficiente, 
   dilo explícitamente.
5. Responde siempre en español.
6. Cita las fuentes cuando sea posible."""

memoria = MemorySaver()

agente = create_react_agent(
    model=llm,
    tools=herramientas,
    prompt=PROMPT_SISTEMA,
    checkpointer=memoria,
)

print("Agente completo listo.")

## 5.4 Conversar con el agente

In [ ]:
config = {"configurable": {"thread_id": "sesion-taller"}}


def chat(mensaje: str):
    """Envía un mensaje al agente y muestra la respuesta."""
    resultado = agente.invoke(
        {"messages": [("human", mensaje)]},
        config=config,
    )
    # Mostrar el razonamiento del agente
    for msg in resultado["messages"]:
        tipo = msg.__class__.__name__
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [Herramienta] {tc['name']}({str(tc['args'])[:80]})")
        elif tipo == "ToolMessage":
            print(f"  [Resultado] {msg.content[:100]}...")
    
    # Respuesta final
    respuesta = resultado["messages"][-1].content
    print(f"\nAgente: {respuesta}")
    return respuesta

In [ ]:
# Conversación de ejemplo
chat("¿Qué es la UBPD y cuál es su mandato?")

In [ ]:
chat("¿Cuántas personas desaparecidas se han registrado?")

In [ ]:
chat("Si procesamos 200 versiones voluntarias por mes y cada una tiene en promedio 45 páginas, ¿cuántas páginas procesamos al año?")

In [ ]:
chat("Resume todo lo que hemos hablado hasta ahora")

## 5.5 Loop interactivo (opcional)

Un chat interactivo para seguir conversando con el agente.

In [ ]:
# Descomentar para usar el chat interactivo
# print("Chat con el agente investigador (escribe 'salir' para terminar)")
# print("=" * 60)
# while True:
#     entrada = input("\nTú: ")
#     if entrada.lower() in ["salir", "exit", "quit"]:
#         print("¡Hasta luego!")
#         break
#     chat(entrada)

## Recapitulación del taller

En este taller aprendimos a:

1. **Notebook 01**: Conectarnos a Ollama y enviar prompts desde Python
2. **Notebook 02**: Usar LangChain para crear chains, templates y salida estructurada
3. **Notebook 03**: Crear herramientas con `@tool` y agentes con `create_react_agent`
4. **Notebook 04**: Implementar RAG local con embeddings y FAISS
5. **Notebook 05**: Integrar todo en un agente completo con memoria

### Próximos pasos
- Experimentar con diferentes modelos de Ollama
- Agregar más herramientas (acceso a APIs, bases de datos, web scraping)
- Persistir la base vectorial en disco para no recalcular embeddings
- Explorar LangGraph para flujos de agentes más complejos
- Desplegar el agente como API con FastAPI o como interfaz con Streamlit